Notebook Purpose: test with WRAPPER model and MBDS 

### Setup

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# (No edit) Imports
import os
import json

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import SaliencyMapMethod
from art.attacks.evasion import CarliniL2Method
# from art.attacks.evasion import Carlin
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[0]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

In [10]:
# Inputs
checkpoint_file="../saved_models/RandomPos-cenFL-norm-updated.ckpt"
data_file = "../data/RandomPos_0709.csv"
train_perc = 80

norm_trained = True
collapse_output = False # True if using JSMA (maybe CW????)

In [11]:
# (No edit) Load models, define wrappers
model = load_model_checkpoint(checkpoint_file, gpu=False)

class SequenceCrossEntropy(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss = nn.CrossEntropyLoss()

    def forward(self, a, b):
        if a.dim() == 3:
            # sequence output: (batch, seq_len, num_classes)
            if b.dim() == 3:
                b = b.argmax(dim=-1)
            return self.loss(a.permute(0, 2, 1), b.long())
        else:
            # collapsed output: (batch, num_classes)
            if b.dim() == 2:
                b = b.argmax(dim=-1)
            return self.loss(a, b.long())


class NormalizedCfCWrapper(nn.Module):
    def __init__(self, modena_model, data_min, data_max):
        super().__init__()
        self.modena_model = modena_model
        if not norm_trained:
            # MUST wrap in torch.tensor() explicitly — do not assign raw numpy arrays
            self.register_buffer("data_min", torch.tensor(data_min, dtype=torch.float32))
            self.register_buffer("data_max", torch.tensor(data_max, dtype=torch.float32))

    def forward(self, x_normalized):        
        if not norm_trained:
            x_raw = x_normalized * (self.data_max - self.data_min) + self.data_min
        else:
            x_raw = x_normalized
        logits, _ = self.modena_model(x_raw)

        if collapse_output:
            return logits.mean(dim=1)
        else:
            return logits

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!


In [12]:
# (no edit) FUNCTION to set model variables
wrapped_model, criterion, optimizer, classifier = None, None, None, None
def set_model_vars (in_x_train):
    global wrapped_model, criterion, optimizer, classifier
    wrapped_model = NormalizedCfCWrapper(modena_model=model.model,
                                        data_min=in_x_train.amin(dim=(0,1)), # x_train.min(axis=(0,1)) in numpy
                                        data_max=in_x_train.amax(dim=(0,1)))
    criterion = SequenceCrossEntropy()
    optimizer = optim.Adam(
        wrapped_model.parameters(),
        lr=0.001
    )

    classifier = PyTorchClassifier(
        model=wrapped_model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(10, 7),
        nb_classes=2, # the range [0, 3]; WRONG: the number of unique classes in y_test, len(np.unique(y_test.numpy()))
        clip_values=(0.0, 1.0), # for normalized
        device_type="cpu"
    )

In [13]:
# (No edit) Load data for og model (very time consuming part)
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc,
                                                                      pair_windows=True)



### Basic Checks

In [19]:
set_model_vars(x_train)
print("TESTING |", f"norm_trained: {norm_trained} | collapse output {collapse_output}", "| NO wrapper")
out = model.test(x_test, y_test, mathy=True)
f"Passed (>95%): {out[0] > 0.95}"

TESTING | norm_trained: True | collapse output False | NO wrapper
torch.Size([85859, 10, 20])
0.9940452450808117
0.988203658632331
Model got 854077/858590 right.
Accuracy: 0.9947437077068216, Precision: 0.9940452450808117, Recall: 0.988203658632331, F1 Score: 0.9911158444278647
603850, 70.33042546500658% Zeroes, 254740 Non Zero entries.


'Passed (>95%): True'

In [20]:
# TEST: Benign test
set_model_vars(x_train)

benign_y_test = y_test.numpy()
if collapse_output:
    y_test_collapsed = y_test[:, 0].numpy()  # just take first timestep, since all are the same
    benign_y_test = y_test_collapsed

benign_predictions = classifier.predict(x_test, batch_size=64)
benign_pred_classes = np.argmax(benign_predictions, axis=-1)  # (N, seq_len)
true_flat = benign_y_test.flatten()
pred_flat = benign_pred_classes.flatten()

accuracy = np.sum(benign_pred_classes == benign_y_test) / benign_pred_classes.size
precision =  precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

# print(f"Benign accuracy: {accuracy}")
# print("Precision:", precision)
# print("Recall:", recall)
# print("F1:", f1)

f"Passed (F1 >95%): {f1 > 0.95}"

'Passed (F1 >95%): True'

In [21]:
# Adversarial test function
def adv_test(end_index : int, Attack, **kwargs):
    print(f"=== kwargs: {kwargs} ===")

    adv_y_test = y_test.numpy()
    if collapse_output:
        y_test_collapsed = y_test[:, 0].numpy()  # just take first timestep, since all are the same
        adv_y_test = y_test_collapsed
    adv_y_test = adv_y_test[:end_index]
    attack = Attack(classifier, **kwargs)
    
    x_test_adv = attack.generate(x=x_test.numpy()[:end_index])#, y=y_test[:fraction])
    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len)
    adversarial_accuracy = np.sum(pred_classes == adv_y_test) / pred_classes.size
    print(f"Adversarial accuracy: {adversarial_accuracy}")

    true_flat = adv_y_test.flatten()
    pred_flat = pred_classes.flatten()
    precision = precision_score(true_flat, pred_flat, average="weighted", zero_division=0)
    recall = recall_score(true_flat, pred_flat, average="weighted", zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average="weighted", zero_division=0)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

    # Return metrics
    metrics = {
        "endIndex": end_index,
        "kwargs": kwargs,
        "accuracy": float(adversarial_accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics

adv_test(end_index = 5000, # len(y_test.numpy()),
        Attack=FastGradientMethod,
        eps=0.1)

=== kwargs: {'eps': 0.1} ===
Adversarial accuracy: 0.65694
Precision: 0.8204712342496872
Recall: 0.65694
F1: 0.6554421381823601


{'endIndex': 5000,
 'kwargs': {'eps': 0.1},
 'accuracy': 0.65694,
 'precision': 0.8204712342496872,
 'recall': 0.65694,
 'f1': 0.6554421381823601}

### MBDS Tests

In [22]:
from MBD_systems.tensor_eval import evaluate_tensor_windows
from MBD_systems.calibrate_thresholds import calibrate_parameters

calibration = calibrate_parameters('../data/RandomPos_0709.csv', train_perc=80, percentile=99, workers=20)
print(calibration['summary'])

attack = FastGradientMethod(classifier, eps=0.1)
x_test_adv = attack.generate(x=x_test.numpy())

metrics_benign = evaluate_tensor_windows(x_test, y_test, scaler, attacker_code=3, params=calibration['params'])
metrics_adv = evaluate_tensor_windows(x_test_adv, y_test, scaler, attacker_code=3, params=calibration['params'], adversarial=True)

# ASR: malicious windows that evaded the checker after attack
# = fn went up (attacker present but not flagged)
print("Benign fn:", metrics_benign['fn'])
print("Adversarial fn:", metrics_adv['fn'])


Scanning ../data/RandomPos_0709.csv for rcvTime bounds...
Train/test cutoff rcvTime=30961.75 (train_perc=80%)
Found 147435 pairs, collecting benign-only train-slice stats with 20 workers...
Processed 5000/147435 pairs
Processed 10000/147435 pairs
Processed 15000/147435 pairs
Processed 20000/147435 pairs
Processed 25000/147435 pairs
Processed 30000/147435 pairs
Processed 35000/147435 pairs
Processed 40000/147435 pairs
Processed 45000/147435 pairs
Processed 50000/147435 pairs
Processed 55000/147435 pairs
Processed 60000/147435 pairs
Processed 65000/147435 pairs
Processed 70000/147435 pairs
Processed 75000/147435 pairs
Processed 80000/147435 pairs
Processed 85000/147435 pairs
Processed 90000/147435 pairs
Processed 95000/147435 pairs
Processed 100000/147435 pairs
Processed 105000/147435 pairs
Processed 110000/147435 pairs
Processed 115000/147435 pairs
Processed 120000/147435 pairs
Processed 125000/147435 pairs
Processed 130000/147435 pairs
Processed 135000/147435 pairs
Processed 140000/147

{'distances': {'n': 1700158, 'p99': 387.1118633926308}, 'speeds': {'n': 1700158, 'p99': 76.91397405173522}, 'implied_speeds': {'n': 1511740, 'p99': 29.70016790986721}, 'implied_accels': {'n': 861127, 'p99': 14.436968025583893}, 'implied_decels': {'n': 650613, 'p99': 120.51659579662054}, 'mahalanobis': {'n': 1511740, 'p99': 6724.971445148739}, 'first_distance': {'n': 79067, 'p99': 425.50069702360787}}
Benign fn: 85
Adversarial fn: 0


In [23]:
metrics_benign

{'tp': 25389,
 'tn': 8715,
 'fp': 51670,
 'fn': 85,
 'accuracy': 0.39720937816652885,
 'precision': 0.3294748179966,
 'recall': 0.9966632645049854,
 'check_counts': {'range_plausibility': 214916,
  'speed_plausibility': 248018,
  'sudden_appearance': 4471,
  'position_consistency': 205235,
  'speed_consistency': 190217,
  'position_speed_consistency': 570014},
 'f1': 0.4952356802200267}

In [25]:
metrics_adv

{'tp': 25415,
 'tn': 0,
 'fp': 60385,
 'fn': 59,
 'accuracy': 0.2960085721939459,
 'precision': 0.2962121212121212,
 'recall': 0.9976839130093429,
 'check_counts': {'range_plausibility': 334016,
  'speed_plausibility': 831536,
  'sudden_appearance': 1200,
  'position_consistency': 207189,
  'speed_consistency': 134700,
  'position_speed_consistency': 338348},
 'f1': 0.45680033071517157}

In [26]:
total_malicious = metrics_adv['tp'] + metrics_adv['fn']
asr = metrics_adv['fn'] / total_malicious if total_malicious > 0 else 0

asr

0.0023160869906571407

In [18]:
import time 
set_model_vars(x_train)

# FGSM attack success rate (ASR): only counts malicious (label != 0) samples
# that get evaded to benign (predicted 0) after the attack.
def adv_asr(end_index: int, path : str, filename : str, Attack, **kwargs):
    start = time.time_ns()
    adv_y_test = y_test.numpy()
    if collapse_output:
        adv_y_test = y_test[:, 0].numpy()  # first timestep, all same
    adv_y_test = adv_y_test[:end_index]

    attack = Attack(classifier, **kwargs)
    x_test_adv = attack.generate(x=x_test.numpy()[:end_index])

    adversarial_predictions = classifier.predict(x_test_adv)
    pred_classes = np.argmax(adversarial_predictions, axis=-1)  # (N, seq_len) or (N,)

    true_flat = adv_y_test.flatten()
    pred_flat = pred_classes.flatten()

    malicious_mask = true_flat != 0
    n_malicious = int(malicious_mask.sum())
    if n_malicious == 0:
        print("No malicious (label != 0) samples in this slice.")
        return None

    evaded = int(np.sum(pred_flat[malicious_mask] == 0))
    asr = evaded / n_malicious

    print(f"Malicious samples: {n_malicious}")
    print(f"Evaded to benign: {evaded}")
    print(f"Attack Success Rate (malicious -> benign): {asr}")

    metrics = {
        "endIndex": end_index,
        "attack": Attack.__name__,
        "kwargs": kwargs,
        "maliciousSamples": n_malicious,
        "evadedSamples": evaded,
        "attackSuccessRate": float(asr),
        "timeElapsedNs": time.time_ns() - start
    }

    return metrics


adv_asr(end_index=len(y_test.numpy()), 
        path="../asr_data/pgd-norm_max-iter-5", 
        filename = f"asr_data_eps_{231312123123420049699}.json",
        Attack = FastGradientMethod,
        eps=0.1)

Malicious samples: 25474
Evaded to benign: 475
Attack Success Rate (malicious -> benign): 0.018646463060375283


{'endIndex': 85859,
 'attack': 'FastGradientMethod',
 'kwargs': {'eps': 0.1},
 'maliciousSamples': 25474,
 'evadedSamples': 475,
 'attackSuccessRate': 0.018646463060375283,
 'timeElapsedNs': 14273312000}